# Optuna Hyperparameter Sweeping & Multi-GPU Training 🚀

Bienvenue dans ce tutoriel interactif. La structure du projet a été repensée avec **Hydra** et sa combinaison d'extensions **Optuna Sweeper** pour gérer parfaitement l'orchestration complexe qu'est l'optimisation des hyperparamètres, et avec **PyTorch Lightning** pour gérer l'exécution matérielle massive (DDP, CPU/GPU, TPUs...).

Ce notebook vous explique comment comprendre l'architecture, explorer l'espace de recherche et déclencher des exécutions depuis la console !

## 1. Comment fonctionne le Sweep Optuna avec Hydra ?

Plutôt que d'écrire des scripts manuels pour tester le taux d'apprentissage ou la taille du batch, nous avons configuré les paramètres dans le fichier `configs/config.yaml` :

```yaml
hydra:
  sweeper:
    sampler:
      seed: ${seed}
    direction: maximize  # Optuna cherchera à maximiser la précision `val_acc` retournée par train.py
    n_trials: 10
    params:
      lr: choice(0.0001, 0.001, 0.005)              # Optuna testera parmi ces 3 LR
      optimizer_name: choice("adam", "sgd")         # Testera l'impact de l'optimiseur
      batch_size: choice(16, 32, 64)                # Impactera la mémoire VRAM de la carte graphique
```

Toute commande d'entraînement standard s'exécute avec `python train.py`. 
Pour **déclencher une optimisation Optuna**, il suffit d'utiliser le flag Multi-Run de Hydra `--multirun` (ou `-m`).

## 2. Multi-GPU et DDP (Distributed Data Parallel)

L'argument `trainer.strategy` est configuré sur "auto". PyTorch Lightning basculera automatiquement sur `ddp` (DistributedDataParallel) de manière complètement transparente. Vos batchs seront divisés intelligemment sur l'ensemble de vos accélérateurs matériels, augmentant la taille totale effective du batch global.

Si vous avez deux GPUs sur votre système, Hydra surchargera l'instruction et injectera les arguments `trainer.devices=2` et optionnellement `trainer.strategy=ddp`.

## 3. Démonstration de lancement Formel

La commande ci-dessous est tout ce dont vous aurez besoin pour réaliser un balayage Optuna (Sweeping) sur les hyperparamètres listés, tout en orchestrant plusieurs cartes graphiques NVIDIA. 

*(Dé-commentez le point d'exclamation pour l'exécuter localement, ou plus recommandable, lancez là directement depuis un bash de tmux ou un terminal)*

In [ ]:
# ⚠️ Assurez-vous d'avoir `hydra-optuna-sweeper` installé (pip install hydra-optuna-sweeper)

# Exécution en Local / Multi-run : Optuna généra 'n_trials' (par ex. 10 runs) et isolera 
# chaque logging/modèle dans les dossiers 'experiments/bcs_determination/multirun_<time>/...'

# -> !python train.py -m \
# ->   trainer.accelerator=gpu \
# ->   trainer.devices=2 \
# ->   trainer.strategy=ddp \
# ->   model_name="vit" \
# ->   max_epochs=20

print("Exécutez la cellule ci-dessus depuis un terminal standard pour ne pas interrompre l'interface Jupyter.")

### Explications des arguments de la commande :
- `-m` : L'activation d'Optuna. Le script au lieu de faire un seul entraînement, utilisera les paramètres définis par Optuna.
- `trainer.accelerator=gpu` : Spécifie que le hardware ciblé est la carte graphique.
- `trainer.devices=2` : Exploite spécifiquement 2 GPUs (A adapter selon votre architecture).
- `trainer.strategy=ddp` : Spécifie textuellement le Data Parallel Distributed.
- `model_name="vit"` : Indiquera qu'on utilise le modèle State-of-the-Art *Swin/Vision-Transformer* ajouté à notre LightningModule au lieu de ResNet50.

## Que se passe-t-il après le Sweep ?
Hydra va créer pour vous le dossier `experiments/bcs_determination/multirun_{date}`. A l'intérieur, un sous dossier (numéroté `0`, `1`, `2`...) existera pour chaque test Optuna. À la toute fin, Hydra imprimera textuellement sur console les *Meilleurs hyperparamètres* ayant obtenu la précision de validation maximale !